In [1]:
from pathlib import Path
from typing import Any, List, Dict, Optional, Literal
import asyncio
import json
import os
import re

from pydantic import BaseModel, Field, ValidationError, field_validator

In [2]:
BASE_DIR = Path.cwd().parent

In [3]:
# Instalacion desactivada para ejecucion directa del notebook.
# Ejecuta manualmente solo si falta algun paquete:
# %pip install -q agent-framework azure-identity python-dotenv nest_asyncio langfuse

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Microsoft Agent Framework migration setup

This notebook now routes LLM calls through Microsoft Agent Framework (`agent-framework`).

Environment options:
- Foundry mode: set `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_MODEL` (and run `az login`).
- OpenAI mode: set `OPENAI_API_KEY` (used by `OpenAIChatClient` fallback).

In [4]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

COLLECTION_NAME = "agent_arena_knowledge_base"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
client = QdrantClient(host="localhost", port=6333)

print("COLLECTION_NAME:", COLLECTION_NAME)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Qdrant client ready")

c:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


COLLECTION_NAME: agent_arena_knowledge_base
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Qdrant client ready


C:\Users\Usuario\AppData\Local\Temp\ipykernel_45468\266652535.py:8: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(host="localhost", port=6333)


In [20]:
def format_context_block(contexts: list[dict]) -> str:
    """
    Format retrieved contexts into a grounded context block for agents.
    Each block keeps a context_id so the agent can cite its evidence.
    """
    blocks = []

    for ctx in contexts:
        block = f"""[{ctx['context_id']}]
Provider: {ctx['provider']}
Document type: {ctx['document_type']}
Source: {ctx['source_file']}
Section: {ctx['section_path']}

{ctx['chunk_text']}"""
        blocks.append(block)

    return "\n\n" + ("\n\n" + "-" * 100 + "\n\n").join(blocks)

In [23]:
def build_provider_filter(provider: str | None = None):
    """
    Build Qdrant filter by provider.
    
    provider can be:
    - None
    - "azure"
    - "aws"
    - "neutral"
    """
    if provider is None:
        return None

    from qdrant_client.models import FieldCondition, Filter, MatchValue

    return Filter(
        must=[
            FieldCondition(
                key="provider",
                match=MatchValue(value=provider)
            )
        ]
    )


def retrieve_contexts(
    query: str,
    provider: str | None = None,
    top_k: int = 5
) -> list[dict]:
    """
    Retrieve relevant contexts from Qdrant using the current query_points API.
    """
    query_vector = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).tolist()

    search_filter = build_provider_filter(provider)

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=search_filter,
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    contexts = []

    for point in response.points:
        payload = dict(point.payload)
        payload["score"] = point.score
        contexts.append(payload)

    return contexts

In [5]:
from dotenv import load_dotenv
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential

try:
    from langfuse import Langfuse
except Exception:
    Langfuse = None

load_dotenv()


# Prefer Microsoft Agent Framework clients. If Foundry env vars are present, use Foundry.
# Otherwise fall back to OpenAI provider via OPENAI_API_KEY for local compatibility.
def _build_agent_client(model: str):
    foundry_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    foundry_model = os.getenv("FOUNDRY_MODEL") or model

    if foundry_endpoint:
        return FoundryChatClient(
            project_endpoint=foundry_endpoint,
            model=foundry_model,
            credential=AzureCliCredential(),
        )

    return OpenAIChatClient(model=model)


def _build_langfuse_client():
    if Langfuse is None:
        return None

    public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
    secret_key = os.getenv("LANGFUSE_SECRET_KEY")
    host = os.getenv("LANGFUSE_HOST")

    if not public_key or not secret_key:
        return None

    try:
        return Langfuse(
            public_key=public_key,
            secret_key=secret_key,
            host=host,
        )
    except Exception as exc:
        print(f"Langfuse disabled due to initialization error: {exc}")
        return None


LANGFUSE_CLIENT = _build_langfuse_client()


def _lf_start_trace(name: str, input_data: Any = None, metadata: Optional[dict] = None):
    if LANGFUSE_CLIENT is None:
        return None
    try:
        return LANGFUSE_CLIENT.trace(
            name=name,
            input=input_data,
            metadata=metadata or {},
        )
    except Exception:
        return None


def _lf_start_span(parent: Any, name: str, input_data: Any = None, metadata: Optional[dict] = None):
    if parent is None:
        return None

    span_fn = getattr(parent, "span", None)
    if callable(span_fn):
        try:
            return span_fn(name=name, input=input_data, metadata=metadata or {})
        except Exception:
            pass

    generation_fn = getattr(parent, "generation", None)
    if callable(generation_fn):
        try:
            return generation_fn(name=name, input=input_data, metadata=metadata or {})
        except Exception:
            pass

    return None


def _lf_update(
    obs: Any,
    *,
    metadata: Optional[dict] = None,
    tags: Optional[list[str]] = None,
    input_data: Any = None,
    output_data: Any = None,
 ):
    if obs is None:
        return

    update_fn = getattr(obs, "update", None)
    if not callable(update_fn):
        return

    payload = {}
    if metadata is not None:
        payload["metadata"] = metadata
    if tags is not None:
        payload["tags"] = tags
    if input_data is not None:
        payload["input"] = input_data
    if output_data is not None:
        payload["output"] = output_data

    if not payload:
        return

    try:
        update_fn(**payload)
    except Exception:
        pass


def _lf_end(obs: Any, output_data: Any = None, error: Optional[Exception] = None):
    if obs is None:
        return

    end_fn = getattr(obs, "end", None)
    if not callable(end_fn):
        return

    payload = {}
    if output_data is not None:
        payload["output"] = output_data
    if error is not None:
        payload["status_message"] = str(error)

    try:
        end_fn(**payload)
    except TypeError:
        try:
            end_fn(output_data)
        except Exception:
            pass
    except Exception:
        pass


def _lf_flush():
    if LANGFUSE_CLIENT is None:
        return
    flush_fn = getattr(LANGFUSE_CLIENT, "flush", None)
    if callable(flush_fn):
        try:
            flush_fn()
        except Exception:
            pass


def _stringify_agent_response(response: Any) -> str:
    if hasattr(response, "messages") and getattr(response, "messages", None):
        text_parts = [m.text for m in response.messages if getattr(m, "text", None)]
        if text_parts:
            return "\n".join(text_parts).strip()
    if isinstance(response, list):
        return "\n".join(_stringify_agent_response(item) for item in response).strip()
    return str(response).strip()


def _create_workflow_backed_agent(
    *,
    model: str,
    name: str,
    instructions: str,
    description: Optional[str] = None,
) -> Agent:
    # Keep the model/provider layer on Microsoft Agent Framework, but avoid wrapping
    # one-shot calls in a nested workflow-as-agent. That pattern triggers notebook timeout issues.
    return Agent(
        client=_build_agent_client(model=model),
        name=name,
        instructions=instructions,
        description=description,
    )


async def call_llm_async(
    prompt: str,
    model: str = "gpt-4o-mini",
    trace: Any = None,
    span_name: str = "llm_call",
    metadata: Optional[dict] = None,
    instructions: Optional[str] = None,
    agent_name: str = "single_shot_agent",
) -> str:
    agent = _create_workflow_backed_agent(
        model=model,
        name=agent_name,
        instructions=instructions or "You are a precise architecture assistant. Follow the user prompt strictly.",
    )

    llm_span = _lf_start_span(
        parent=trace,
        name=span_name,
        input_data=prompt,
        metadata={"model": model, "runtime": "agent_framework_agent", **(metadata or {})},
    )

    try:
        response = await agent.run(prompt)
        result_text = _stringify_agent_response(response)
        _lf_end(llm_span, output_data=result_text)
        return result_text
    except Exception as exc:
        _lf_end(llm_span, error=exc)
        raise


def _run_coroutine_in_new_thread(coro):
    import threading

    result_holder = {}
    error_holder = {}

    def runner():
        try:
            result_holder["value"] = asyncio.run(coro)
        except Exception as exc:
            error_holder["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()

    if "error" in error_holder:
        raise error_holder["error"]

    return result_holder["value"]


def call_llm(prompt: str, model: str = "gpt-4o-mini") -> str:
    # Notebook-safe sync wrapper for async Agent Framework APIs.
    return _run_coroutine_in_new_thread(call_llm_async(prompt=prompt, model=model))

c:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
c:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langfuse\api\core\pydantic_utilities.py:27: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.datetime_parse import parse_date as parse_date


In [50]:
GROUNDED_RULES = """
You are a grounded cloud architecture agent.

STRICT RULES:
1. You must only use information explicitly present in the provided context.
2. Do not introduce services, components, benefits, risks, trade-offs, or alternatives unless they appear in the context.
3. Every recommended component must cite at least one context_id using the format [CTX-0001].
4. If the context is insufficient to justify a component, do not recommend it.
5. If something important is missing, add it under "Missing context".
6. Do not rely on general knowledge.
7. Do not mention unsupported services.
8. Do not fabricate citations.
9. Use only the context IDs that appear in the provided context.
"""


def build_azure_agent_prompt(context_pack: Dict) -> str:
    return f"""
{GROUNDED_RULES}

ROLE:
You are the Azure Architecture Agent.

TASK:
Propose an Azure-native architecture for the user's project using only the retrieved context.

USER PROJECT IDEA:
{context_pack["user_idea"]}

RETRIEVED CONTEXT:
{context_pack["azure_context_block"]}

OUTPUT FORMAT:
Return your answer in Markdown with exactly these sections:

# Azure Architecture Proposal

## 1. Executive summary
Briefly summarize the proposed architecture. Cite context IDs.

## 2. Recommended components
For each component, include:
- Component name
- Role in the architecture
- Why it fits this project
- Evidence: context IDs

Use this format:

### Component: <name>
Role:
Why:
Evidence: [CTX-XXXX]

## 3. Proposed flow
Describe the flow step by step.
Each step must reference evidence when it introduces a component.

## 4. Trade-offs
Only include trade-offs explicitly supported by the context.
Each trade-off must cite evidence.

## 5. MVP approach
Explain the local MVP if supported by context.
Cite evidence.

## 6. Missing context
List any important missing information that would be needed for a stronger proposal.

IMPORTANT:
If the context does not support a claim, do not include it.
"""

In [51]:
def build_aws_agent_prompt(context_pack: Dict) -> str:
    return f"""
{GROUNDED_RULES}

ROLE:
You are the AWS Architecture Agent.

TASK:
Propose an AWS-native architecture for the user's project using only the retrieved context.

USER PROJECT IDEA:
{context_pack["user_idea"]}

RETRIEVED CONTEXT:
{context_pack["aws_context_block"]}

OUTPUT FORMAT:
Return your answer in Markdown with exactly these sections:

# AWS Architecture Proposal

## 1. Executive summary
Briefly summarize the proposed architecture. Cite context IDs.

## 2. Recommended components
For each component, include:
- Component name
- Role in the architecture
- Why it fits this project
- Evidence: context IDs

Use this format:

### Component: <name>
Role:
Why:
Evidence: [CTX-XXXX]

## 3. Proposed flow
Describe the flow step by step.
Each step must reference evidence when it introduces a component.

## 4. Trade-offs
Only include trade-offs explicitly supported by the context.
Each trade-off must cite evidence.

## 5. MVP approach
Explain the local MVP if supported by context.
Cite evidence.

## 6. Missing context
List any important missing information that would be needed for a stronger proposal.

IMPORTANT:
If the context does not support a claim, do not include it.
"""

In [52]:
def get_valid_context_ids(context_pack: Dict, agent: str) -> set:
    if agent == "azure":
        contexts = context_pack["azure_contexts"] + context_pack["neutral_contexts"]
    elif agent == "aws":
        contexts = context_pack["aws_contexts"] + context_pack["neutral_contexts"]
    else:
        contexts = (
            context_pack["azure_contexts"]
            + context_pack["aws_contexts"]
            + context_pack["neutral_contexts"]
        )

    return {ctx["context_id"] for ctx in contexts}


def extract_cited_context_ids(text: str) -> set:
    return set(re.findall(r"CTX-\d{4}", text))

In [53]:
def validate_citations(proposal: str, valid_context_ids: set) -> Dict:
    cited_ids = extract_cited_context_ids(proposal)
    invalid_ids = cited_ids - valid_context_ids
    missing = len(cited_ids) == 0

    return {
        "cited_ids": sorted(cited_ids),
        "invalid_ids": sorted(invalid_ids),
        "has_citations": not missing,
        "valid": len(invalid_ids) == 0 and not missing,
    }

In [54]:
def build_judge_prompt(
    user_idea: str,
    azure_proposal: str,
    aws_proposal: str,
) -> str:
    return f"""
You are a grounded architecture judge.

STRICT RULES:
1. Compare only the two proposals provided.
2. Do not introduce new cloud services or new architecture components.
3. Do not use general knowledge.
4. If a comparison cannot be made from the proposals, say so.
5. Keep the original citations from the proposals when referencing a claim.

USER PROJECT IDEA:
{user_idea}

AZURE PROPOSAL:
{azure_proposal}

AWS PROPOSAL:
{aws_proposal}

TASK:
Compare both proposals and produce a final recommendation.

OUTPUT FORMAT:

# Architecture Comparison

## 1. Executive recommendation
Recommend Azure, AWS, or Local MVP first.
Justify only using the proposals.

## 2. Azure strengths
Use only claims from the Azure proposal.

## 3. AWS strengths
Use only claims from the AWS proposal.

## 4. Key trade-offs
Only include trade-offs already present in the proposals.

## 5. Recommended next step
Give the next practical step for the project.
Do not introduce unsupported services.
"""

In [26]:
class RetrievalQueries(BaseModel):
    azure: List[str] = Field(default_factory=list)
    aws: List[str] = Field(default_factory=list)
    neutral: List[str] = Field(default_factory=list)


class PlannerOutput(BaseModel):
    project_summary: str
    project_type: str

    required_capabilities: List[str] = Field(default_factory=list)
    non_functional_requirements: List[str] = Field(default_factory=list)

    explicit_constraints: List[str] = Field(default_factory=list)
    inferred_constraints: List[str] = Field(default_factory=list)
    assumptions: List[str] = Field(default_factory=list)

    explicit_cloud_preferences: List[str] = Field(default_factory=list)
    missing_information: List[str] = Field(default_factory=list)
    retrieval_focus: List[str] = Field(default_factory=list)

    retrieval_queries: RetrievalQueries

    @field_validator(
        "required_capabilities",
        "non_functional_requirements",
        "explicit_constraints",
        "inferred_constraints",
        "assumptions",
        "explicit_cloud_preferences",
        "missing_information",
        "retrieval_focus",
        mode="before"
    )
    @classmethod
    def ensure_list(cls, value):
        if value is None:
            return []
        if isinstance(value, str):
            return [value]
        return value

In [27]:
Provider = Literal["azure", "aws", "neutral", "unknown"]
DocumentType = Literal[
    "cloud_reference",
    "service_reference",
    "architecture_pattern",
    "decision_record",
    "project_case",
    "unknown"
]


class BusinessContext(BaseModel):
    organization_name: str = "Internal AI Architecture Team"
    department_or_area: str = "AI, Automation and Cloud Architecture"
    business_domain: str = (
        "Enterprise AI solutions, document processing, PMO support and cloud architecture advisory"
    )
    target_users: List[str] = Field(default_factory=list)
    current_priorities: List[str] = Field(default_factory=list)
    preferred_working_style: List[str] = Field(default_factory=list)
    architecture_principles: List[str] = Field(default_factory=list)
    known_constraints: List[str] = Field(default_factory=list)
    preferred_clouds: List[str] = Field(default_factory=list)
    notes: str = ""


DEFAULT_BUSINESS_CONTEXT = BusinessContext(
    organization_name="Internal AI Architecture Team",
    department_or_area="AI, Automation and Cloud Architecture",
    business_domain=(
        "Enterprise AI solutions focused on document processing, cloud architecture, "
        "multi-agent systems, PMO use cases, RAG systems and automation workflows."
    ),
    target_users=[
        "technical teams",
        "PMO teams",
        "business stakeholders",
        "cloud architecture teams",
    ],
    current_priorities=[
        "compare Azure and AWS architectures",
        "justify technical decisions with evidence",
        "avoid hallucinated architecture components",
        "produce proposals reusable in real projects",
        "prefer clear MVP-to-production evolution",
    ],
    preferred_working_style=[
        "structured architecture proposals",
        "explicit trade-offs",
        "grounded recommendations",
        "clear explanation of why each service is selected",
    ],
    architecture_principles=[
        "start with the simplest architecture that satisfies requirements",
        "separate MVP architecture from production architecture",
        "use managed services when operational overhead should be reduced",
        "use local or open-source components when experimentation is priority",
        "every recommended component must be justified by retrieved context",
    ],
    known_constraints=[],
    preferred_clouds=["azure", "aws"],
    notes=(
        "The system acts as an architecture advisor. It should not only list services. "
        "It should explain why a service or pattern fits the specific project."
    ),
)


class RetrievedContext(BaseModel):
    context_id: str
    chunk_id: Optional[str] = None

    provider: Provider
    document_type: DocumentType

    source_file: str
    source_path: Optional[str] = None
    document_title: Optional[str] = None

    section_title: Optional[str] = None
    section_path: str

    chunk_text: str
    contextualized_chunk_text: Optional[str] = None

    score: Optional[float] = None


class ContextPack(BaseModel):
    user_idea: str
    business_context: BusinessContext

    planner_output: PlannerOutput
    final_queries: RetrievalQueries

    azure_contexts: List[RetrievedContext]
    aws_contexts: List[RetrievedContext]
    neutral_contexts: List[RetrievedContext]

    azure_context_block: str
    aws_context_block: str

In [28]:
class CitationValidation(BaseModel):
    cited_ids: List[str] = Field(default_factory=list)
    invalid_ids: List[str] = Field(default_factory=list)
    has_citations: bool
    valid: bool


class AgentArenaResult(BaseModel):
    context_pack: ContextPack

    azure_proposal: str
    aws_proposal: str

    azure_validation: CitationValidation
    aws_validation: CitationValidation

    final_comparison: str
    final_architecture_proposal: str
    full_report: str

In [29]:
def build_requirement_extractor_prompt(
    user_idea: str,
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
) -> str:
    return f"""
You are a cloud architecture requirement extraction agent.

Your task is to analyze the user's project idea and extract structured architectural requirements.

BUSINESS CONTEXT:
{business_context.model_dump_json(indent=2)}

CRITICAL RULES:
1. Return only a JSON object.
2. Do not return a JSON Schema.
3. Do not include "$defs", "properties", "required", "title", or "type".
4. Do not include Markdown.
5. Do not include explanations outside JSON.
6. Do not propose a final architecture.
7. Do not choose specific cloud services unless the user explicitly mentions them.
8. Do not invent requirements that are not reasonably implied by the user idea.
9. Do not add cost constraints unless the user explicitly mentions cost, budget, free tier, avoiding payment, or local-only execution.
10. Do not add local-first constraints unless the user explicitly says the MVP must run locally or without cloud.
11. If the user says the solution must be hosted in Azure or AWS from the beginning, do not include local_first or avoid_paid_cloud_resources.
12. Separate explicit constraints from inferred constraints.
13. Retrieval queries will be used to search a local knowledge base.

USER PROJECT IDEA:
{user_idea}

Return exactly this JSON structure:

{{
  "project_summary": "short summary of the project",
  "project_type": "short project type label",
  "required_capabilities": [
    "capability_1",
    "capability_2"
  ],
  "non_functional_requirements": [
    "requirement_1",
    "requirement_2"
  ],
  "explicit_constraints": [
    "constraints explicitly stated by the user"
  ],
  "inferred_constraints": [
    "constraints reasonably inferred from the user idea"
  ],
  "assumptions": [
    "assumptions made because the user did not provide enough detail"
  ],
  "explicit_cloud_preferences": [
    "azure",
    "aws"
  ],
  "missing_information": [
    "missing information needed to design a stronger architecture"
  ],
  "retrieval_focus": [
    "focus area 1",
    "focus area 2"
  ],
  "retrieval_queries": {{
    "azure": [
      "query for Azure knowledge base retrieval"
    ],
    "aws": [
      "query for AWS knowledge base retrieval"
    ],
    "neutral": [
      "query for neutral architecture patterns and project cases"
    ]
  }}
}}

Allowed capability labels include:
- document_storage
- document_upload
- document_ingestion
- text_extraction
- document_intelligence
- chunking
- semantic_retrieval
- vector_search
- hybrid_search
- metadata_filtering
- llm_answer_generation
- multi_agent_reasoning
- architecture_comparison
- judge_agent
- final_architecture_synthesis
- cloud_deployment
- event_driven_processing
- async_processing
- observability
- authentication
- cost_control
- dashboarding
- data_extraction
- data_validation
- bi_reporting
- api_layer
- containerized_deployment
- serverless_processing

Examples of explicit constraints:
- must be deployed in Azure
- must be deployed in AWS
- must run locally
- must avoid paid cloud resources
- must use retrieved context only
- must compare Azure and AWS

Important:
Only include a constraint if it is explicitly stated or strongly implied.
Do not add avoid_paid_cloud_resources unless cost avoidance is clearly present.
Do not add local_first unless local execution is clearly present.
Return the filled JSON object itself.
"""

In [30]:
def extract_json_string_from_text(text: str) -> str:
    """
    Extract a JSON object string from LLM output.
    """
    text = text.strip()

    if text.startswith("{") and text.endswith("}"):
        return text

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM output:\n{text}")

    return match.group(0)


def repair_schema_like_output(data: dict) -> dict:
    """
    Some LLMs mistakenly return a JSON Schema-like object and place payload in properties.
    """
    if "properties" in data and isinstance(data["properties"], dict):
        properties = data["properties"]

        expected_keys = {
            "project_summary",
            "project_type",
            "required_capabilities",
            "non_functional_requirements",
            "explicit_constraints",
            "inferred_constraints",
            "assumptions",
            "explicit_cloud_preferences",
            "missing_information",
            "retrieval_focus",
            "retrieval_queries",
        }

        if any(key in properties for key in expected_keys):
            return properties

    return data


def parse_planner_output(raw_llm_output: str) -> PlannerOutput:
    """
    Parse and validate planner output using Pydantic.
    Includes fallback repair if the LLM returned a schema-like object.
    """
    json_text = extract_json_string_from_text(raw_llm_output)

    try:
        data = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Invalid JSON returned by planner:\n{json_text}"
        ) from e

    data = repair_schema_like_output(data)

    try:
        return PlannerOutput.model_validate(data)
    except ValidationError as e:
        raise ValueError(
            f"Planner output does not match PlannerOutput schema:\n{e}\n\nRepaired data:\n{json.dumps(data, indent=2, ensure_ascii=False)}\n\nRaw output:\n{raw_llm_output}"
        ) from e

In [31]:
async def extract_requirements_with_llm_async(
    user_idea: str,
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
    debug: bool = False,
 ) -> PlannerOutput:
    prompt = build_requirement_extractor_prompt(
        user_idea=user_idea,
        business_context=business_context,
    )
    raw_output = await call_llm_async(prompt)

    if debug:
        print("RAW LLM OUTPUT")
        print("=" * 120)
        print(raw_output)

    planner_output = parse_planner_output(raw_output)
    return planner_output


def extract_requirements_with_llm(
    user_idea: str,
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
    debug: bool = False,
 ) -> PlannerOutput:
    return _run_coroutine_in_new_thread(
        extract_requirements_with_llm_async(
            user_idea=user_idea,
            business_context=business_context,
            debug=debug,
        )
    )

In [ ]:
# Celda de ejemplo/planner desactivada para flujo directo.
# Define tu query en la celda de ejecucion principal (mas abajo).

In [32]:
def ensure_non_empty_queries(
    planner_output: PlannerOutput
) -> RetrievalQueries:
    """
    Ensure every provider has at least one query.
    """
    queries = planner_output.retrieval_queries

    azure = queries.azure or [
        "Azure architecture for document extraction, BI reporting, multi-agent decisions and service selection rationale"
    ]

    aws = queries.aws or [
        "AWS architecture for document extraction, BI reporting, multi-agent decisions and service selection rationale"
    ]

    neutral = queries.neutral or [
        "architecture patterns, project cases, and decision rationale for document to BI pipelines"
    ]

    return RetrievalQueries(
        azure=azure,
        aws=aws,
        neutral=neutral,
    )


def build_final_retrieval_queries(
    user_idea: str,
    planner_output: PlannerOutput,
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
) -> RetrievalQueries:
    raw_queries = ensure_non_empty_queries(planner_output)

    capabilities_text = ", ".join(planner_output.required_capabilities)
    explicit_constraints_text = ", ".join(planner_output.explicit_constraints)
    inferred_constraints_text = ", ".join(planner_output.inferred_constraints)
    assumptions_text = ", ".join(planner_output.assumptions)
    retrieval_focus_text = ", ".join(planner_output.retrieval_focus)

    business_context_text = business_context.model_dump_json(indent=2)

    azure_query = f"""
User project idea:
{user_idea}

Business context:
{business_context_text}

Extracted capabilities:
{capabilities_text}

Explicit constraints:
{explicit_constraints_text}

Inferred constraints:
{inferred_constraints_text}

Assumptions:
{assumptions_text}

Retrieval focus:
{retrieval_focus_text}

Azure retrieval queries:
{chr(10).join("- " + q for q in raw_queries.azure)}

Retrieve Azure service references, Azure decision records, Azure project cases and Azure architecture patterns that are explicitly relevant.
"""

    aws_query = f"""
User project idea:
{user_idea}

Business context:
{business_context_text}

Extracted capabilities:
{capabilities_text}

Explicit constraints:
{explicit_constraints_text}

Inferred constraints:
{inferred_constraints_text}

Assumptions:
{assumptions_text}

Retrieval focus:
{retrieval_focus_text}

AWS retrieval queries:
{chr(10).join("- " + q for q in raw_queries.aws)}

Retrieve AWS service references, AWS decision records, AWS project cases and AWS architecture patterns that are explicitly relevant.
"""

    neutral_query = f"""
User project idea:
{user_idea}

Business context:
{business_context_text}

Extracted capabilities:
{capabilities_text}

Explicit constraints:
{explicit_constraints_text}

Inferred constraints:
{inferred_constraints_text}

Assumptions:
{assumptions_text}

Retrieval focus:
{retrieval_focus_text}

Neutral retrieval queries:
{chr(10).join("- " + q for q in raw_queries.neutral)}

Retrieve neutral architecture patterns, prior project cases, decision records and reusable implementation patterns.
"""

    return RetrievalQueries(
        azure=[azure_query],
        aws=[aws_query],
        neutral=[neutral_query],
    )

In [36]:
# Prueba intermedia desactivada para ejecucion de corrido.
# (esta celda dependia de planner_output y podia fallar en Run All).

NameError: name 'planner_output' is not defined

In [13]:
def validate_retrieved_contexts(raw_contexts: List[dict]) -> List[RetrievedContext]:
    contexts = []

    for raw in raw_contexts:
        try:
            contexts.append(RetrievedContext.model_validate(raw))
        except ValidationError as e:
            raise ValueError(
                f"Retrieved context does not match schema:\n{e}\n\nRaw context:\n{raw}"
            ) from e

    return contexts


def retrieve_typed_contexts(
    query: str,
    provider: Provider,
    top_k: int
) -> List[RetrievedContext]:
    raw_contexts = retrieve_contexts(
        query=query,
        provider=provider,
        top_k=top_k,
    )

    return validate_retrieved_contexts(raw_contexts)

In [64]:
# Validacion/inspeccion de retrieval desactivada para flujo directo de query.

NameError: name 'final_queries' is not defined

In [14]:
def format_context_block_typed(contexts: List[RetrievedContext]) -> str:
    blocks = []

    for ctx in contexts:
        block = f"""[{ctx.context_id}]
Provider: {ctx.provider}
Document type: {ctx.document_type}
Source: {ctx.source_file}
Section: {ctx.section_path}

{ctx.chunk_text}"""
        blocks.append(block)

    return "\n\n" + ("\n\n" + "-" * 100 + "\n\n").join(blocks)

In [66]:
# Inspeccion de contexto desactivada para flujo directo de query.

NameError: name 'azure_contexts' is not defined

In [33]:
async def build_dynamic_context_pack_with_llm_planner_async(
    user_idea: str,
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
    top_k_provider: int = 8,
    top_k_neutral: int = 6
) -> ContextPack:
    planner_output = await extract_requirements_with_llm_async(
        user_idea=user_idea,
        business_context=business_context,
    )

    final_queries = build_final_retrieval_queries(
        user_idea=user_idea,
        planner_output=planner_output,
        business_context=business_context,
    )

    azure_contexts = retrieve_typed_contexts(
        query=final_queries.azure[0],
        provider="azure",
        top_k=top_k_provider,
    )

    aws_contexts = retrieve_typed_contexts(
        query=final_queries.aws[0],
        provider="aws",
        top_k=top_k_provider,
    )

    neutral_contexts = retrieve_typed_contexts(
        query=final_queries.neutral[0],
        provider="neutral",
        top_k=top_k_neutral,
    )

    azure_context_block = format_context_block_typed(
        azure_contexts + neutral_contexts
    )

    aws_context_block = format_context_block_typed(
        aws_contexts + neutral_contexts
    )

    return ContextPack(
        user_idea=user_idea,
        business_context=business_context,
        planner_output=planner_output,
        final_queries=final_queries,
        azure_contexts=azure_contexts,
        aws_contexts=aws_contexts,
        neutral_contexts=neutral_contexts,
        azure_context_block=azure_context_block,
        aws_context_block=aws_context_block,
    )


def build_dynamic_context_pack_with_llm_planner(
    user_idea: str,
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
    top_k_provider: int = 8,
    top_k_neutral: int = 6
) -> ContextPack:
    return _run_coroutine_in_new_thread(
        build_dynamic_context_pack_with_llm_planner_async(
            user_idea=user_idea,
            business_context=business_context,
            top_k_provider=top_k_provider,
            top_k_neutral=top_k_neutral,
        )
    )

In [68]:
# Validacion manual del context_pack desactivada para flujo directo de query.

RuntimeError: Timeout should be used inside a task

In [16]:
GROUNDED_RULES = """
You are a grounded cloud architecture agent.

STRICT RULES:
1. You must only use information explicitly present in the provided context.
2. Do not introduce services, components, benefits, risks, trade-offs, or alternatives unless they appear in the context.
3. Every recommended component must cite at least one context_id using the format [CTX-0001].
4. If the context is insufficient to justify a component, do not recommend it.
5. If something important is missing, add it under "Missing context".
6. Do not rely on general knowledge.
7. Do not mention unsupported services.
8. Do not fabricate citations.
9. Use only the context IDs that appear in the provided context.
"""

In [34]:
def build_azure_agent_prompt_typed(context_pack: ContextPack) -> str:
    return f"""
{GROUNDED_RULES}

ROLE:
You are the Azure Architecture Agent.

PERSONA:
You are a senior Azure cloud architect working for the organization described in the business context.
You design enterprise-grade AI, document processing, RAG and automation architectures.
You must think like a practical solution architect: justify every component, separate MVP from production, and avoid unnecessary complexity.

BUSINESS CONTEXT:
{context_pack.business_context.model_dump_json(indent=2)}

TASK:
Propose an Azure-native architecture for the user's project using only the retrieved context.

USER PROJECT IDEA:
{context_pack.user_idea}

LLM-EXTRACTED REQUIREMENTS:
{context_pack.planner_output.model_dump_json(indent=2)}

RETRIEVED CONTEXT:
{context_pack.azure_context_block}

OUTPUT FORMAT:
Return your answer in Markdown with exactly these sections:

# Azure Architecture Proposal

## 1. Executive summary

## 2. Recommended components
For each component, include:
- Component name
- Role in the architecture
- Why it fits this project
- Evidence: context IDs

Use this format:

### Component: <name>
Role:
Why:
Evidence: [CTX-XXXX]

## 3. Proposed flow

## 4. Trade-offs

## 5. MVP approach

## 6. Missing context

IMPORTANT:
If the context does not support a claim, do not include it.
"""


def build_aws_agent_prompt_typed(context_pack: ContextPack) -> str:
    return f"""
{GROUNDED_RULES}

ROLE:
You are the AWS Architecture Agent.

PERSONA:
You are a senior AWS cloud architect working for the organization described in the business context.
You design enterprise-grade AI, document processing, RAG and automation architectures.
You must think like a practical solution architect: justify every component, separate MVP from production, and avoid unnecessary complexity.

BUSINESS CONTEXT:
{context_pack.business_context.model_dump_json(indent=2)}

TASK:
Propose an AWS-native architecture for the user's project using only the retrieved context.

USER PROJECT IDEA:
{context_pack.user_idea}

LLM-EXTRACTED REQUIREMENTS:
{context_pack.planner_output.model_dump_json(indent=2)}

RETRIEVED CONTEXT:
{context_pack.aws_context_block}

OUTPUT FORMAT:
Return your answer in Markdown with exactly these sections:

# AWS Architecture Proposal

## 1. Executive summary

## 2. Recommended components
For each component, include:
- Component name
- Role in the architecture
- Why it fits this project
- Evidence: context IDs

Use this format:

### Component: <name>
Role:
Why:
Evidence: [CTX-XXXX]

## 3. Proposed flow

## 4. Trade-offs

## 5. MVP approach

## 6. Missing context

IMPORTANT:
If the context does not support a claim, do not include it.
"""

In [18]:
def get_valid_context_ids_typed(
    context_pack: ContextPack,
    agent: Literal["azure", "aws", "all"]
) -> set[str]:
    if agent == "azure":
        contexts = context_pack.azure_contexts + context_pack.neutral_contexts
    elif agent == "aws":
        contexts = context_pack.aws_contexts + context_pack.neutral_contexts
    else:
        contexts = (
            context_pack.azure_contexts +
            context_pack.aws_contexts +
            context_pack.neutral_contexts
        )

    return {ctx.context_id for ctx in contexts}


def extract_cited_context_ids(text: str) -> set[str]:
    return set(re.findall(r"CTX-\d{4}", text))


def validate_citations_typed(
    proposal: str,
    valid_context_ids: set[str]
) -> CitationValidation:
    cited_ids = extract_cited_context_ids(proposal)
    invalid_ids = cited_ids - valid_context_ids
    has_citations = len(cited_ids) > 0

    return CitationValidation(
        cited_ids=sorted(cited_ids),
        invalid_ids=sorted(invalid_ids),
        has_citations=has_citations,
        valid=len(invalid_ids) == 0 and has_citations,
    )

In [46]:
def build_rewrite_prompt_typed(
    original_proposal: str,
    valid_context_ids: set[str],
    context_block: str,
    agent_name: str
) -> str:
    valid_ids_text = ", ".join(sorted(valid_context_ids))

    return f"""
You are revising a grounded architecture proposal.

The previous proposal used invalid or missing citations.

STRICT RULES:
1. Use only the context provided below.
2. Use only these valid context IDs:
{valid_ids_text}
3. Every recommended component must cite at least one valid context ID.
4. Remove any unsupported service, claim, benefit, risk, or trade-off.
5. If the context does not support something, move it to "Missing context".

AGENT:
{agent_name}

VALID CONTEXT:
{context_block}

PREVIOUS PROPOSAL:
{original_proposal}

Rewrite the proposal in Markdown.
"""


def build_final_rewrite_prompt_typed(
    original_proposal: str,
    valid_context_ids: set[str],
    context_block: str,
) -> str:
    valid_ids_text = ", ".join(sorted(valid_context_ids))

    return f"""
You are revising the final architecture proposal to ensure end-to-end traceability.

STRICT RULES:
1. Keep the same architecture decision unless the available evidence forces a correction.
2. Use only the context provided below.
3. Use only these valid context IDs:
{valid_ids_text}
4. Add explicit citations [CTX-XXXX] to every key claim, service decision, trade-off, and roadmap step.
5. Do not introduce services, claims, or risks not supported by the context.
6. Preserve Spanish language and executive style.

VALID CONTEXT:
{context_block}

PREVIOUS FINAL PROPOSAL:
{original_proposal}

Rewrite the final proposal in Markdown, preserving the same section structure.
"""

In [47]:
def build_judge_prompt_typed(
    context_pack: ContextPack,
    azure_proposal: str,
    aws_proposal: str
) -> str:
    return f"""
You are a grounded architecture judge.

STRICT RULES:
1. Compare only the two proposals provided.
2. Do not introduce new cloud services or new architecture components.
3. Do not use general knowledge.
4. If a comparison cannot be made from the proposals, say so.
5. Keep the original citations from the proposals when referencing a claim.

BUSINESS CONTEXT:
{context_pack.business_context.model_dump_json(indent=2)}

USER PROJECT IDEA:
{context_pack.user_idea}

LLM-EXTRACTED REQUIREMENTS:
{context_pack.planner_output.model_dump_json(indent=2)}

AZURE PROPOSAL:
{azure_proposal}

AWS PROPOSAL:
{aws_proposal}

TASK:
Compare both proposals and produce a final recommendation.

OUTPUT FORMAT:

# Architecture Comparison

## 1. Executive recommendation

## 2. Azure strengths

## 3. AWS strengths

## 4. Key trade-offs

## 5. Recommended next step
"""


def build_final_architecture_prompt_typed(
    context_pack: ContextPack,
    azure_proposal: str,
    aws_proposal: str,
    final_comparison: str,
) -> str:
    return f"""
You are the Final Architecture Agent.

ROLE:
You are a senior solution architect. Your task is to produce the final architecture proposal after reviewing:
- the Azure proposal,
- the AWS proposal,
- the judge comparison.

BUSINESS CONTEXT:
{context_pack.business_context.model_dump_json(indent=2)}

USER PROJECT IDEA:
{context_pack.user_idea}

LLM-EXTRACTED REQUIREMENTS:
{context_pack.planner_output.model_dump_json(indent=2)}

AZURE PROPOSAL:
{azure_proposal}

AWS PROPOSAL:
{aws_proposal}

JUDGE COMPARISON:
{final_comparison}

STRICT RULES:
1. Produce one final architecture proposal.
2. Choose Azure, AWS, or a phased or hybrid recommendation only if justified by the previous proposals and judge comparison.
3. Do not introduce new services or components that did not appear in the Azure proposal, AWS proposal or judge comparison.
4. Preserve context citations when referencing claims.
5. Every key claim, major service choice, trade-off, and roadmap step must include at least one citation in format [CTX-XXXX].
6. The output should be directly usable as an architecture proposal.
7. Do not output a debate. Output the selected final architecture.
8. If the available evidence is insufficient to choose one provider, recommend the safest next step and explain what information is missing.
9. Write the complete output in Spanish.
10. Keep it executive and concise, target 450 to 700 words.
11. Use practical and decision-oriented language.

OUTPUT FORMAT:

# Propuesta Final de Arquitectura

## 1. Recomendacion ejecutiva

## 2. Por que se selecciono esta opcion

## 3. Arquitectura objetivo

## 4. Flujo end-to-end

## 5. Version MVP

## 6. Version produccion

## 7. Riesgos y trade-offs

## 8. Informacion faltante

## 9. Roadmap de implementacion
"""

In [48]:
async def run_agent_arena_with_llm_planner_pydantic_async(
    user_idea: str,
    model: str = "gpt-4o-mini",
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
) -> AgentArenaResult:
    """
    Full Agent Arena pipeline using Microsoft Agent Framework and Langfuse tracing.
    """

    run_trace = _lf_start_trace(
        name="agent_arena_run",
        input_data={"user_idea": user_idea, "model": model},
        metadata={
            "pipeline": "dynamic_llm_planner_pydantic",
            "runtime": "microsoft_agent_framework",
            "tracing": "langfuse_optional",
        },
    )

    try:
        context_pack_span = _lf_start_span(
            parent=run_trace,
            name="build_context_pack",
            input_data={"user_idea": user_idea},
        )
        context_pack = await build_dynamic_context_pack_with_llm_planner_async(
            user_idea=user_idea,
            business_context=business_context,
        )
        _lf_end(
            context_pack_span,
            output_data={
                "azure_contexts": len(context_pack.azure_contexts),
                "aws_contexts": len(context_pack.aws_contexts),
                "neutral_contexts": len(context_pack.neutral_contexts),
            },
        )

        planner = context_pack.planner_output
        cloud_prefs = planner.explicit_cloud_preferences or []
        merged_constraints = set(
            planner.explicit_constraints + planner.inferred_constraints
        )
        business_tags = [
            "pipeline:agent_arena",
            f"project_type:{planner.project_type or 'unknown'}",
            f"azure_pref:{'azure' in cloud_prefs}",
            f"aws_pref:{'aws' in cloud_prefs}",
            f"local_first:{'local_first' in merged_constraints}",
        ]
        for cap in planner.required_capabilities[:3]:
            business_tags.append(f"cap:{cap}")

        _lf_update(
            run_trace,
            metadata={
                "project_summary": planner.project_summary,
                "project_type": planner.project_type,
                "required_capabilities": planner.required_capabilities,
                "explicit_constraints": planner.explicit_constraints,
                "inferred_constraints": planner.inferred_constraints,
                "assumptions": planner.assumptions,
                "explicit_cloud_preferences": planner.explicit_cloud_preferences,
            },
            tags=business_tags,
        )

        prompt_span = _lf_start_span(
            parent=run_trace,
            name="build_agent_prompts",
            input_data={"user_idea": user_idea},
        )
        azure_prompt = build_azure_agent_prompt_typed(context_pack)
        aws_prompt = build_aws_agent_prompt_typed(context_pack)
        _lf_end(
            prompt_span,
            output_data={
                "azure_prompt_chars": len(azure_prompt),
                "aws_prompt_chars": len(aws_prompt),
            },
        )

        proposal_span = _lf_start_span(
            parent=run_trace,
            name="generate_cloud_proposals",
            metadata={"execution": "concurrent"},
        )
        azure_task = call_llm_async(
            prompt=azure_prompt,
            model=model,
            trace=run_trace,
            span_name="azure_agent_generation",
            metadata={"agent": "azure"},
            agent_name="azure_architecture_agent",
        )
        aws_task = call_llm_async(
            prompt=aws_prompt,
            model=model,
            trace=run_trace,
            span_name="aws_agent_generation",
            metadata={"agent": "aws"},
            agent_name="aws_architecture_agent",
        )
        azure_proposal, aws_proposal = await asyncio.gather(azure_task, aws_task)
        _lf_end(
            proposal_span,
            output_data={
                "azure_proposal_chars": len(azure_proposal),
                "aws_proposal_chars": len(aws_proposal),
            },
        )

        validation_input_span = _lf_start_span(
            parent=run_trace,
            name="collect_valid_context_ids",
        )
        azure_valid_ids = get_valid_context_ids_typed(
            context_pack=context_pack,
            agent="azure",
        )
        aws_valid_ids = get_valid_context_ids_typed(
            context_pack=context_pack,
            agent="aws",
        )
        all_valid_ids = get_valid_context_ids_typed(
            context_pack=context_pack,
            agent="all",
        )
        _lf_end(
            validation_input_span,
            output_data={
                "azure_valid_ids": len(azure_valid_ids),
                "aws_valid_ids": len(aws_valid_ids),
                "all_valid_ids": len(all_valid_ids),
            },
        )

        citation_span = _lf_start_span(parent=run_trace, name="citation_validation")
        azure_validation = validate_citations_typed(
            proposal=azure_proposal,
            valid_context_ids=azure_valid_ids,
        )
        aws_validation = validate_citations_typed(
            proposal=aws_proposal,
            valid_context_ids=aws_valid_ids,
        )
        _lf_end(
            citation_span,
            output_data={
                "azure_valid": azure_validation.valid,
                "aws_valid": aws_validation.valid,
                "azure_invalid_ids": azure_validation.invalid_ids,
                "aws_invalid_ids": aws_validation.invalid_ids,
            },
        )

        if not azure_validation.valid:
            rewrite_span = _lf_start_span(parent=run_trace, name="azure_rewrite")
            rewrite_prompt = build_rewrite_prompt_typed(
                original_proposal=azure_proposal,
                valid_context_ids=azure_valid_ids,
                context_block=context_pack.azure_context_block,
                agent_name="Azure Architecture Agent",
            )
            azure_proposal = await call_llm_async(
                prompt=rewrite_prompt,
                model=model,
                trace=run_trace,
                span_name="azure_rewrite_generation",
                metadata={"agent": "azure", "rewrite": True},
                agent_name="azure_rewrite_agent",
            )
            azure_validation = validate_citations_typed(
                proposal=azure_proposal,
                valid_context_ids=azure_valid_ids,
            )
            _lf_end(
                rewrite_span,
                output_data={
                    "azure_valid_after_rewrite": azure_validation.valid,
                    "azure_invalid_ids_after_rewrite": azure_validation.invalid_ids,
                },
            )

        if not aws_validation.valid:
            rewrite_span = _lf_start_span(parent=run_trace, name="aws_rewrite")
            rewrite_prompt = build_rewrite_prompt_typed(
                original_proposal=aws_proposal,
                valid_context_ids=aws_valid_ids,
                context_block=context_pack.aws_context_block,
                agent_name="AWS Architecture Agent",
            )
            aws_proposal = await call_llm_async(
                prompt=rewrite_prompt,
                model=model,
                trace=run_trace,
                span_name="aws_rewrite_generation",
                metadata={"agent": "aws", "rewrite": True},
                agent_name="aws_rewrite_agent",
            )
            aws_validation = validate_citations_typed(
                proposal=aws_proposal,
                valid_context_ids=aws_valid_ids,
            )
            _lf_end(
                rewrite_span,
                output_data={
                    "aws_valid_after_rewrite": aws_validation.valid,
                    "aws_invalid_ids_after_rewrite": aws_validation.invalid_ids,
                },
            )

        judge_prompt = build_judge_prompt_typed(
            context_pack=context_pack,
            azure_proposal=azure_proposal,
            aws_proposal=aws_proposal,
        )
        final_comparison = await call_llm_async(
            prompt=judge_prompt,
            model=model,
            trace=run_trace,
            span_name="judge_generation",
            metadata={"agent": "judge"},
            agent_name="architecture_judge_agent",
        )

        final_architecture_prompt = build_final_architecture_prompt_typed(
            context_pack=context_pack,
            azure_proposal=azure_proposal,
            aws_proposal=aws_proposal,
            final_comparison=final_comparison,
        )
        final_architecture_proposal = await call_llm_async(
            prompt=final_architecture_prompt,
            model=model,
            trace=run_trace,
            span_name="final_architecture_generation",
            metadata={"agent": "final_architecture"},
            agent_name="final_architecture_agent",
        )

        final_validation = validate_citations_typed(
            proposal=final_architecture_proposal,
            valid_context_ids=all_valid_ids,
        )

        if not final_validation.valid:
            final_rewrite_span = _lf_start_span(
                parent=run_trace,
                name="final_architecture_rewrite",
            )
            all_context_block = format_context_block_typed(
                context_pack.azure_contexts
                + context_pack.aws_contexts
                + context_pack.neutral_contexts
            )
            final_rewrite_prompt = build_final_rewrite_prompt_typed(
                original_proposal=final_architecture_proposal,
                valid_context_ids=all_valid_ids,
                context_block=all_context_block,
            )
            final_architecture_proposal = await call_llm_async(
                prompt=final_rewrite_prompt,
                model=model,
                trace=run_trace,
                span_name="final_architecture_rewrite_generation",
                metadata={"agent": "final_architecture", "rewrite": True},
                agent_name="final_architecture_rewrite_agent",
            )
            final_validation = validate_citations_typed(
                proposal=final_architecture_proposal,
                valid_context_ids=all_valid_ids,
            )
            _lf_end(
                final_rewrite_span,
                output_data={
                    "final_valid_after_rewrite": final_validation.valid,
                    "final_invalid_ids_after_rewrite": final_validation.invalid_ids,
                    "final_cited_ids_after_rewrite": final_validation.cited_ids,
                },
            )

        full_report = final_architecture_proposal

        result = AgentArenaResult(
            context_pack=context_pack,
            azure_proposal=azure_proposal,
            aws_proposal=aws_proposal,
            azure_validation=azure_validation,
            aws_validation=aws_validation,
            final_comparison=final_comparison,
            final_architecture_proposal=final_architecture_proposal,
            full_report=full_report,
        )

        _lf_update(
            run_trace,
            metadata={
                "azure_valid": result.azure_validation.valid,
                "aws_valid": result.aws_validation.valid,
                "azure_invalid_ids": result.azure_validation.invalid_ids,
                "aws_invalid_ids": result.aws_validation.invalid_ids,
                "final_valid": final_validation.valid,
                "final_invalid_ids": final_validation.invalid_ids,
                "final_cited_ids": final_validation.cited_ids,
            },
            tags=[
                f"azure_valid:{result.azure_validation.valid}",
                f"aws_valid:{result.aws_validation.valid}",
                f"final_valid:{final_validation.valid}",
            ],
        )

        _lf_end(
            run_trace,
            output_data={
                "status": "success",
                "azure_valid": result.azure_validation.valid,
                "aws_valid": result.aws_validation.valid,
                "final_valid": final_validation.valid,
            },
        )
        _lf_flush()
        return result

    except Exception as exc:
        _lf_end(run_trace, error=exc)
        _lf_flush()
        raise


def run_agent_arena_with_llm_planner_pydantic(
    user_idea: str,
    model: str = "gpt-4o-mini",
    business_context: BusinessContext = DEFAULT_BUSINESS_CONTEXT,
) -> AgentArenaResult:
    """Sync wrapper to keep existing notebook cells compatible."""
    return _run_coroutine_in_new_thread(
        run_agent_arena_with_llm_planner_pydantic_async(
            user_idea=user_idea,
            model=model,
            business_context=business_context,
        )
    )

In [51]:
user_idea = """
Quiero construir una herramienta de gestión de iniciativas. Esta herramienta debe brindar la posibilidad de dar de alta una iniciativa y poder acompañar todo el proceso de su desarrollo hasta la puesta en producción. La idea es que sirva para gestionar iniciativas de IA. También debe gestionar el buzón del departamento para poder hacer ese seguimineto del desarrollo y de su post-producción.
"""

result = await run_agent_arena_with_llm_planner_pydantic_async(user_idea)

print(result.full_report)

# Propuesta Final de Arquitectura

## 1. Recomendacion ejecutiva
Después de analizar tanto la propuesta de Azure como la de AWS, se recomienda implementar la arquitectura basada en **Azure** para la herramienta de gestión de iniciativas de IA. Esta elección se fundamenta en sus capacidades para manejar datos estructurados, robustas características de monitoreo y el soporte para estilos de trabajo más complejos, lo cual puede ser beneficioso en el contexto empresarial.

## 2. Por que se selecciono esta opcion
La decisión se apoya en las fortalezas de Azure, particularmente en el uso de **Azure SQL Database** para el almacenamiento de datos estructurados que es crucial para el seguimiento detallado de iniciativas (evidencia: [CTX-0051]). Además, la inclusión de **Application Insights** proporciona capacidades de monitoreo que garantizan la disponibilidad operativa durante el desarrollo y la postproducción. A pesar de la complejidad del **Microsoft Agent Framework**, ofrece la flexibilida

In [42]:
# Smoke test desactivado para ejecucion directa del notebook.

Longitud ejecutiva OK: 680 palabras.
Smoke test final_architecture_proposal: OK


In [57]:
# Validaciones de citas desactivadas para ejecucion directa del notebook.

Azure validation
{
  "cited_ids": [
    "CTX-0082",
    "CTX-0083",
    "CTX-0115",
    "CTX-0116",
    "CTX-0126"
  ],
  "invalid_ids": [],
  "has_citations": true,
  "valid": true
}

AWS validation
{
  "cited_ids": [
    "CTX-0100",
    "CTX-0101"
  ],
  "invalid_ids": [],
  "has_citations": true,
  "valid": true
}


In [58]:
# Dump detallado de contextos desactivado para flujo directo de query.

Azure contexts
----------------------------------------------------------------------------------------------------
CTX-0116 | azure | decision_record | azure_ai_search_for_enterprise_rag.md | Decision Record: Azure AI Search for Enterprise RAG > Requirements
CTX-0082 | azure | cloud_reference | azure_functions_event_ingestion.md | Azure Functions for Event-Driven Document Ingestion > Typical usage in RAG architectures
CTX-0053 | azure | cloud_reference | azure_agentic_app.md | Azure Solution: Multi-Agent Architecture Advisor > Architecture
CTX-0083 | azure | cloud_reference | azure_functions_event_ingestion.md | Azure Functions for Event-Driven Document Ingestion > Role in a document assistant architecture
CTX-0126 | azure | decision_record | azure_ai_search_for_enterprise_rag.md | Decision Record: Azure AI Search for Enterprise RAG > When to use this decision
CTX-0115 | azure | decision_record | azure_ai_search_for_enterprise_rag.md | Decision Record: Azure AI Search for Enterprise R

In [53]:
# Guardado automatico de artefactos desactivado para ejecucion directa.
# Si quieres exportar resultados luego, reactiva esta celda.

Saved report: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\agent_outputs\agent_arena_dynamic_llm_planner_pydantic_report.md
Saved JSON result: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\agent_outputs\agent_arena_dynamic_llm_planner_pydantic_result.json


In [39]:
# Reimpresion final desactivada (ya se imprime en la celda principal).

# Final Architecture Proposal

## 1. Recommended architecture
The recommended architecture for the PDF document processing platform is based on **Azure services**, specifically utilizing Azure Blob Storage, Azure Cognitive Services (Form Recognizer), Azure SQL Database, Azure App Service, and Application Insights.

## 2. Why this option was selected
Azure provides a cohesive integration of services tailored for document processing, particularly with Azure Cognitive Services, which simplifies text extraction from diverse PDF formats. Additionally, the streamlined monitoring capabilities of Application Insights contribute to high availability and performance monitoring. The proposal aligns with the organization's objective to justify technical decisions based on evidence, with Azure's components fitting well into the requirements for scalability, high availability, and data security.

## 3. Target architecture
- **Document Storage**: Azure Blob Storage
- **Text Extraction**: Azure Cognit

In [50]:
import re


def _ids_from_text(text: str) -> list[str]:
    return sorted(set(re.findall(r"CTX-\d{4}", text or "")))


def _ctx_map_from_result(res: AgentArenaResult) -> dict[str, RetrievedContext]:
    all_ctx = (
        list(res.context_pack.azure_contexts)
        + list(res.context_pack.aws_contexts)
        + list(res.context_pack.neutral_contexts)
    )
    return {ctx.context_id: ctx for ctx in all_ctx}


def _print_trace(label: str, ids: list[str], ctx_map: dict[str, RetrievedContext]) -> None:
    print(f"\n{label}: {len(ids)} citas")
    if not ids:
        print("- Sin citas explicitas")
        return

    for cid in ids:
        ctx = ctx_map.get(cid)
        if ctx is None:
            print(f"- {cid} | NO_ENCONTRADO_EN_CONTEXTO")
            continue
        print(
            f"- {cid} | provider={ctx.provider} | doc_type={ctx.document_type} | source={ctx.source_file} | section={ctx.section_path}"
        )


ctx_map = _ctx_map_from_result(result)

azure_ids = _ids_from_text(result.azure_proposal)
aws_ids = _ids_from_text(result.aws_proposal)
final_ids = _ids_from_text(result.final_architecture_proposal)

print("Trazabilidad de evidencia por etapa")
_print_trace("Azure proposal", azure_ids, ctx_map)
_print_trace("AWS proposal", aws_ids, ctx_map)
_print_trace("Final architecture proposal", final_ids, ctx_map)

print("\nResumen rapido")
print(f"- Azure citas: {len(azure_ids)}")
print(f"- AWS citas: {len(aws_ids)}")
print(f"- Final citas: {len(final_ids)}")

Trazabilidad de evidencia por etapa

Azure proposal: 3 citas
- CTX-0051 | provider=azure | doc_type=service_reference | source=azure_agentic_app.md | section=Azure Solution: Multi-Agent Architecture Advisor > Recommended Azure resources
- CTX-0055 | provider=azure | doc_type=service_reference | source=azure_agentic_app.md | section=Azure Solution: Multi-Agent Architecture Advisor > Scalable version
- CTX-0056 | provider=azure | doc_type=service_reference | source=azure_agentic_app.md | section=Azure Solution: Multi-Agent Architecture Advisor > Pros

AWS proposal: 2 citas
- CTX-0040 | provider=aws | doc_type=service_reference | source=aws_serverless_app.md | section=AWS Solution: Serverless AI Architecture Advisor with Lambda
- CTX-0042 | provider=aws | doc_type=service_reference | source=aws_serverless_app.md | section=AWS Solution: Serverless AI Architecture Advisor with Lambda > Recommended AWS resources

Final architecture proposal: 2 citas
- CTX-0051 | provider=azure | doc_type=ser